# FarrahAI — Notebook 2: Topic Classification

This notebook demonstrates:
- Building TF-IDF features from extracted note chunks
- Training Logistic Regression, Random Forest, and XGBoost
- Comparing all classifiers (accuracy, precision, recall, F1)
- Confusion matrix visualization
- Saving best model for production use


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from modules.ml_models import (
    build_tfidf_features, train_topic_classifier,
    compare_classifiers, save_model
)
print('Ready ✓')

## Step 1: Load your labeled dataset

Format: CSV with columns `text` and `topic`

You can create this by:
1. Running OCR on your notes
2. Manually labeling each chunk with its topic/unit

Even 50-100 labeled examples is enough for a demo.


In [ ]:
# Option A: Load from CSV
# df = pd.read_csv('../data/processed/labeled_chunks.csv')

# Option B: Use this synthetic example to test the pipeline
# Replace with your actual data before the presentation!
synthetic_data = [
    # AI/ML topics
    ("Backpropagation computes gradients using chain rule to update neural network weights", "Neural Networks"),
    ("Gradient descent minimizes loss by moving in negative gradient direction", "Optimization"),
    ("Overfitting occurs when model memorizes training data and fails to generalize", "Model Evaluation"),
    ("Cross validation splits data into k folds for robust evaluation", "Model Evaluation"),
    ("Decision trees split data recursively using information gain or gini impurity", "Decision Trees"),
    ("Random forest is an ensemble of decision trees using bagging technique", "Ensemble Methods"),
    ("XGBoost uses gradient boosting with regularization to prevent overfitting", "Ensemble Methods"),
    ("Support vector machine finds hyperplane that maximizes margin between classes", "SVM"),
    ("Precision is true positives divided by all predicted positives", "Model Evaluation"),
    ("Recall measures how many actual positives were correctly identified", "Model Evaluation"),
    ("F1 score is harmonic mean of precision and recall for imbalanced classes", "Model Evaluation"),
    ("K means clustering assigns points to nearest centroid iteratively", "Clustering"),
    ("Hierarchical clustering builds tree of clusters using agglomeration or division", "Clustering"),
    ("Reinforcement learning agent learns policy by maximizing cumulative reward", "Reinforcement Learning"),
    ("Q learning uses bellman equation to learn optimal action value function", "Reinforcement Learning"),
    ("Convolutional neural networks use filters to detect spatial features in images", "Neural Networks"),
    ("LSTM networks use gates to control flow of information through time steps", "Neural Networks"),
    ("Logistic regression models probability of binary outcome using sigmoid function", "Classification"),
    ("Naive bayes classifier assumes conditional independence between features", "Classification"),
    ("PCA reduces dimensionality by projecting data onto principal components", "Dimensionality Reduction"),
    ("Autoencoders learn compressed representation of input through encoder decoder", "Deep Learning"),
    ("Batch normalization normalizes layer inputs to stabilize and speed training", "Deep Learning"),
    ("Dropout randomly deactivates neurons during training to prevent overfitting", "Neural Networks"),
    ("Transfer learning reuses pretrained model weights for new tasks", "Deep Learning"),
    ("Attention mechanism allows model to focus on relevant parts of input sequence", "Deep Learning"),
]

df = pd.DataFrame(synthetic_data, columns=['text', 'topic'])
print(f'Dataset size: {len(df)} samples')
print(f'Topics: {df["topic"].nunique()} unique')
print(df['topic'].value_counts())

## Step 2: Build TF-IDF Features

In [ ]:
vectorizer, X = build_tfidf_features(df['text'].tolist(), max_features=3000)
y = df['topic'].tolist()

print(f'Feature matrix: {X.shape}')
print(f'Labels: {len(set(y))} classes')

## Step 3: Compare All Classifiers

In [ ]:
comparison_df = compare_classifiers(X, y)
comparison_df.to_csv('../outputs/classifier_comparison.csv', index=False)
print('\nSaved to outputs/classifier_comparison.csv ✓')

## Step 4: Visualize Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(comparison_df))
width = 0.25

b1 = ax.bar(x - width,   comparison_df['Accuracy'],     width, label='Accuracy',      color='#2ecc71')
b2 = ax.bar(x,           comparison_df['F1 (macro)'],   width, label='F1 (macro)',     color='#3498db')
b3 = ax.bar(x + width,   comparison_df['F1 (weighted)'],width, label='F1 (weighted)',  color='#e67e22')

ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'], fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Classifier Performance Comparison — FarrahAI Topic Classification')
ax.legend()
ax.bar_label(b1, fmt='%.2f', padding=2, fontsize=8)
ax.bar_label(b2, fmt='%.2f', padding=2, fontsize=8)
ax.bar_label(b3, fmt='%.2f', padding=2, fontsize=8)

plt.tight_layout()
plt.savefig('../outputs/classifier_comparison_chart.png', dpi=150)
plt.show()

## Step 5: Confusion Matrix for Best Model (XGBoost)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

le = LabelEncoder()
y_enc = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_enc, test_size=0.2, random_state=42)

xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                    eval_metric='mlogloss', random_state=42)
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)

fig, ax = plt.subplots(figsize=(12, 9))
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title('XGBoost Confusion Matrix — Topic Classification')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix_xgboost.png', dpi=150)
plt.show()

# Save the model
save_model(xgb, '../models/topic_classifier_xgboost.pkl')
save_model(le,  '../models/label_encoder.pkl')
save_model(vectorizer, '../models/tfidf_vectorizer.pkl')
print('Models saved ✓')